# 03 - Data Quality Checks

Validates the Silver layer before Gold is built. Any failed rule raises an exception,
which fails the Notebook activity and stops the pipeline before bad data reaches reporting.

In [ ]:
from pyspark.sql import functions as F

customers = spark.table("silver_customers")
products = spark.table("silver_products")
sales = spark.table("silver_sales")

checks = []

def check(name, failed_rows):
    # Record a rule; failed_rows is the number of rows that break it
    checks.append((name, failed_rows == 0, failed_rows))

def duplicate_keys(df, key):
    return df.groupBy(key).count().filter(F.col("count") > 1).count()

# Uniqueness
check("Unique customer_id", duplicate_keys(customers, "customer_id"))
check("Unique product_id", duplicate_keys(products, "product_id"))
check("Unique order_id", duplicate_keys(sales, "order_id"))

# Completeness and validity (single pass over sales)
s = sales.agg(
    F.sum(F.when(F.col("order_date").isNull(), 1).otherwise(0)).alias("null_dates"),
    F.sum(F.when(F.col("customer_id").isNull() | F.col("product_id").isNull(), 1).otherwise(0)).alias("null_keys"),
    F.sum(F.when(F.col("quantity") <= 0, 1).otherwise(0)).alias("bad_qty"),
    F.sum(F.when(F.col("sales_amount") < 0, 1).otherwise(0)).alias("neg_sales"),
    F.sum(F.when(F.col("discount_pct") < 0, 1).when(F.col("discount_pct") > 1, 1).otherwise(0)).alias("bad_discount"),
    # sales_amount must equal quantity * unit_price * (1 - discount), allowing 1 cent rounding
    F.sum(F.when(
        F.abs(F.col("sales_amount") - F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct"))) > 0.01, 1
    ).otherwise(0)).alias("amount_mismatch"),
).first()

check("No null order_date", s["null_dates"])
check("No null customer/product keys", s["null_keys"])
check("Quantity greater than zero", s["bad_qty"])
check("Non-negative sales_amount", s["neg_sales"])
check("Discount between 0 and 1", s["bad_discount"])
check("sales_amount = qty x price x (1 - discount)", s["amount_mismatch"])

# Referential integrity
check("No orphan customer keys", sales.join(customers, "customer_id", "left_anti").count())
check("No orphan product keys", sales.join(products, "product_id", "left_anti").count())

for name, passed, failed_rows in checks:
    print(f"{'PASS' if passed else 'FAIL'} - {name}" + ("" if passed else f" ({failed_rows:,} rows)"))

failed = [name for name, passed, _ in checks if not passed]
if failed:
    raise Exception(f"Data quality failed: {failed}")
print(f"All {len(checks)} data-quality checks passed.")